# Resume Model V2 — 100k Labeling

Clean rebuild. V1 remains frozen.

**Goal:** create 100,000 resume–job training pairs, compact them for LLM labeling, then train supervised V2 models.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import joblib

PROJECT_DIR = "/content/drive/MyDrive/Resume_Model_Project"
V2_DIR = os.path.join(PROJECT_DIR, "V2_100k")
INPUT_DIR = os.path.join(V2_DIR, "Inputs")
OUTPUT_DIR = os.path.join(V2_DIR, "Outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Drive mounted.")
print("V2 input dir:", INPUT_DIR)
print("V2 output dir:", OUTPUT_DIR)

In [ ]:
# Load the three V1 source artifacts copied into the V2 input folder

candidate_df = pd.read_pickle(os.path.join(INPUT_DIR, "candidate_df.pkl"))
entry_job_df = pd.read_pickle(os.path.join(INPUT_DIR, "entry_job_df.pkl"))
match_vectorizer = joblib.load(os.path.join(INPUT_DIR, "tfidf_vectorizer.pkl"))

print("Candidates:", len(candidate_df))
print("Jobs:", len(entry_job_df))
print("TF-IDF features:", len(match_vectorizer.get_feature_names_out()))

In [ ]:
# Verify the exact structures before generating any pairs

print("Candidate columns:", candidate_df.columns.tolist())
print("Job columns:", entry_job_df.columns.tolist())

assert len(candidate_df) == 1540, "Unexpected candidate count"
assert len(entry_job_df) == 434, "Unexpected job count"
assert len(match_vectorizer.get_feature_names_out()) == 44046, "Unexpected TF-IDF vocabulary size"

required_candidate_cols = {"resume_text", "current_level"}
required_job_cols = {"JobID", "Title", "ExperienceLevel", "YearsOfExperience", "Skills", "Responsibilities", "Keywords", "job_text"}

assert required_candidate_cols.issubset(candidate_df.columns)
assert required_job_cols.issubset(entry_job_df.columns)

print("
✅ V2 source artifacts verified.")
print("✅ No pair generation or labeling has happened yet.")

In [ ]:
# Save a small manifest so the V2 run is reproducible

manifest = {
    "candidates": len(candidate_df),
    "jobs": len(entry_job_df),
    "possible_pairs": len(candidate_df) * len(entry_job_df),
    "tfidf_features": len(match_vectorizer.get_feature_names_out()),
    "status": "source_artifacts_verified"
}

manifest_df = pd.DataFrame([manifest])
manifest_path = os.path.join(OUTPUT_DIR, "v2_run_manifest.csv")
manifest_df.to_csv(manifest_path, index=False)

print(manifest_df.to_string(index=False))
print("Saved:", manifest_path)